# Supporting churn prediction prototype

Predict cancellation in the next calendar month from active month-end features. This notebook trains models, reports held-out performance, and explains why accuracy alone is misleading.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if not (root / 'scripts').exists():
    root = root.parent
assert (root / 'scripts').exists(), 'Open this notebook inside the project'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
from scripts.common import ROOT, AS_OF, read_csv
from IPython.display import display, Image


In [2]:
from scripts.model import modeling_frame, train_model, FEATURES
p=modeling_frame()
display(p[p.split.ne('unused')].groupby('split').agg(rows=('customer_id','size'),customers=('customer_id','nunique'),churn_events=('next_month_churn','sum')))
print('Features:', FEATURES)

,rows,customers,churn_events
split,,,
test,4540,1661,120
train,28208,4122,898
validation,2641,1406,93


Features: ['customer_age_months', 'monthly_revenue', 'support_tickets', 'complaints', 'login_frequency', 'monthly_usage', 'feature_usage_score', 'payment_failures', 'late_payments', 'auto_renew', 'discount_percentage', 'engagement_score', 'recent_usage_decline', 'plan_type', 'contract_type', 'acquisition_channel', 'region', 'payment_method']


## Leakage controls

Train/validation/test customers are disjoint SHA-256 buckets. Training outcomes finish before validation feature months; validation outcomes finish before test feature months. Imputation, scaling and encoding fit only on training rows. Final churn dates are used to create labels, never as features.

In [3]:
sets={name:set(p.loc[p.split.eq(name),'customer_id']) for name in ['train','validation','test']}
assert not (sets['train']&sets['validation'] or sets['train']&sets['test'] or sets['validation']&sets['test'])
summary=train_model()

Model: logistic_regression; test ROC-AUC 0.650; PR-AUC 0.062.


In [4]:
table=pd.DataFrame({name:{k:v for k,v in result['test'].items() if k!='confusion_matrix'} for name,result in summary['models'].items()}).T
display(table)
print('Selected on validation PR-AUC:',summary['selected_model'])

,accuracy,precision,recall,f1,roc_auc,pr_auc,threshold
logistic_regression,0.879736,0.070565,0.291667,0.113636,0.650354,0.062107,0.65
decision_tree,0.741189,0.041703,0.400000,0.075531,0.592373,0.042614,0.62
baseline,0.973568,0.000000,0.000000,0.000000,0.500000,0.026432,0.50


Selected on validation PR-AUC: logistic_regression


## Class imbalance and operational limits

A majority-class baseline can have high accuracy while finding no cancellations. Inspect precision, recall and average precision alongside ROC-AUC. The F1 threshold is selected on validation data; test outcomes do not tune the model. Repeated monthly observations are correlated and scores are uncalibrated.

In [5]:
selected=summary['models'][summary['selected_model']]['test']
display(pd.DataFrame(selected['confusion_matrix'],index=['Actual stay','Actual churn'],columns=['Predicted stay','Predicted churn']))
print('\n'.join(summary['limitations']))

,Predicted stay,Predicted churn
Actual stay,3959,461
Actual churn,85,35


Synthetic data reflects the generator's assumptions.
Repeated customer-months are correlated; no independent-sample confidence intervals are claimed.
Balanced-class scores are not calibrated operational probabilities.
No churn date, final status, final tenure, future revenue or customer ID is a feature.
No deployment, realized impact or production readiness claimed.


## Next work with real data

Validate across later customer cohorts, calibrate probabilities on untouched validation data, evaluate top-k lift at actual outreach capacity, and compare intervention value after contact costs. Use randomized retention experiments to estimate incremental impact.